# Retriever Performance Benchmark (S1/S2/S5/S6/S7)

This notebook measures **retrieval-time cost** (latency) for the retriever systems:
- S1: BM25
- S2: Dense (clause-level → control-level adapter)
- S5: Hybrid (BM25 + Dense via RRF)
- S6: Hybrid + Cross-Encoder rerank
- S7: ComplianceGPT retriever (QUR variants + weighted RRF + safe rerank blending + **rerank gate**)

We report both:
- **S7-gated (typical)**: rerank may be skipped by the gate
- **S7-worst (always rerank)**: gate disabled (upper-bound latency)

**Important:** Index building (BM25/Dense) is *not* included in latency. Only per-query retrieval is timed.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================================
# 0) CONFIG (EDIT PATHS HERE)
# ==========================================================
import os, time, json, math, re
from typing import Any, Dict, List, Tuple, Optional, Iterable
from collections import defaultdict

import numpy as np
import pandas as pd

pd.options.display.float_format = '{:.4f}'.format

# ---- Data paths (Google Drive / Colab) ----

DRIVE_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"

# 1. Catalog Paths (CCS)
REV5_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# 2. Gold Standard Datasets
REV5_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv"

ERROR_BANK_CSV = f"{DRIVE_ROOT}/data/error_bank/error_bank_v1.csv"
# Rewrites (QUR outputs)
QUR_CSV_REV5 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_error_bank.csv"

# Backward-compatible alias used elsewhere in this notebook:
QUR_CSV_ERROR_BANK = QUR_CSV_ERRB

# ---- Select which CCS + QUR set to use for benchmarking ----
DATASET = "rev5"   # one of: {"rev5", "rev4"}  (error_bank is optional for query list only)

# ---- Retriever models ----
DENSE_MODEL_ID = "intfloat/e5-small-v2"
RERANKER_MODEL_ID = "BAAI/bge-reranker-base"

# ---- Core knobs (keep aligned with AblationStudy_S1_8) ----
BM25_K1 = 1.5
BM25_B = 0.75

CANDIDATE_SET_SIZE = 50   # pre-rerank pool size for S6/S7
RRF_K_DEFAULT = 60

# S7 (QUR + weighted RRF)
S7_MAX_REWRITES = 3
S7_REWRITE_WEIGHT = 0.25
S7_REWRITE_JACCARD_MIN = 0.15

# S7 (rerank blending + gate)
S7_RERANK_ALPHA = 0.65
S7_RERANK_APPLY_MIN_MARGIN_RATIO = 0.15   # gate threshold (typical)
S7_RERANK_SKIP_MIN_BASE_MARGIN_RATIO = 0.10  # start permissive for performance benchmark
S7_RERANK_SKIP_REQUIRE_TOP1_AGREEMENT = False  # set True later if accuracy holds
S7_RERANK_SKIP_ENABLED = True

# ---- Logging controls ----
SAVE_RUN_LOG = True  # writes per-run timing rows
RUN_LOG_INCLUDE_S7_META = True  # include S7 gate internals in per-run log

S7_RERANK_APPLY_MIN_MARGIN_RATIO_WORST = 0.0  # disable gate (always rerank)

# ---- Benchmark controls ----
# Use one representative query OR multiple queries.
USE_SINGLE_QUERY = False      # True: only time one query repeatedly; False: sample multiple queries
SINGLE_QUERY_ID = 1           # used when USE_SINGLE_QUERY=True (query_id from QUR CSV)

NUM_QUERIES = 20              # used when USE_SINGLE_QUERY=False (sample from QUR CSV)
REPEATS_PER_QUERY = 5         # repeats per query per system (increase for more stable results)

SAMPLE_SEED = 1337
QUERY_SAMPLE_METHOD = "hash"   # {"hash", "random"}

print("Config loaded.")


Config loaded.


In [3]:
# ==========================================================
# 1) TEXT + CONTROL-ID NORMALIZATION & JSONL LOADERS
# ==========================================================
PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    if not isinstance(text, str): text = "" if pd.isna(text) else str(text)
    s = text.lower().replace('-', ' ')
    s = PUNCT_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def tokenize(text: str) -> List[str]:
    return [t.lower() for t in TOKEN_RE.findall(str(text))]

def normalize_control_id(control_id: str) -> str:
    if not isinstance(control_id, str): return ""
    return control_id.strip().upper().replace("_", "-")

def control_id_aliases(control_id: str) -> str:
    cid = normalize_control_id(control_id)
    return f"{cid} {cid.replace('-', '')} {cid.replace('-', ' ')}"

def load_clause_records_jsonl(jsonl_path: str, keep_kinds=("smt", "gdn")) -> List[Dict]:
    out = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            if keep_kinds and str(rec.get("kind","")).lower() not in keep_kinds: continue

            txt = str(rec.get("text","")).strip()
            if not txt: continue

            out.append({
                "clause_id": str(rec.get("id","")).strip(),
                "control_id": normalize_control_id(rec.get("control_id") or rec.get("control") or ""),
                "title": str(rec.get("title","")).strip(),
                "text": txt,
                "kind": str(rec.get("kind","")).lower()
            })
    return out

def build_control_docs_from_clauses(records):
    by_ctl = defaultdict(list)
    titles = {}
    for r in records:
        cid = r["control_id"]
        if not cid: continue
        by_ctl[cid].append(r["text"])
        if r["title"]: titles[cid] = r["title"]

    docs = []
    for cid, parts in by_ctl.items():
        header = [control_id_aliases(cid)]
        if titles.get(cid): header.append(titles[cid])
        full_text = "\n".join(header + parts)
        docs.append({"control_id": cid, "text": full_text})
    return docs

def build_control_fallback_text_map(records):
    # For reranker fallback
    by_ctl = defaultdict(list)
    for r in records:
        cid = r["control_id"]
        if cid: by_ctl[cid].append(r["text"])
    return {k: "\n".join(v[:8]) for k,v in by_ctl.items()}

# ==========================================================
# 2) BM25 + METRICS
# ==========================================================
class BM25Okapi:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0

        self.tf = []
        df = {}
        for doc in corpus_tokens:
            tf_doc = {}
            for t in doc:
                tf_doc[t] = tf_doc.get(t, 0) + 1
            self.tf.append(tf_doc)
            for t in tf_doc:
                df[t] = df.get(t, 0) + 1

        self.idf = {t: math.log((self.N - freq + 0.5)/(freq + 0.5) + 1) for t, freq in df.items()}

    def get_scores(self, query_tokens):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in query_tokens:
            if t not in self.idf: continue
            idf = self.idf[t]
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(t, 0)
                if f:
                    num = f * (self.k1 + 1)
                    den = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i]/self.avgdl))
                    scores[i] += idf * (num/den)
        return scores

    def get_top_n(self, query_tokens, n=10):
        scores = self.get_scores(query_tokens)
        if n >= len(scores): return np.argsort(-scores).tolist()
        idx = np.argpartition(-scores, n-1)[:n]
        return idx[np.argsort(-scores[idx])].tolist()

def build_bm25(docs):
    ids = [d["control_id"] for d in docs]
    toks = [tokenize(normalize_text(d["text"])) for d in docs]
    return BM25Okapi(toks, k1=BM25_K1, b=BM25_B), ids

# ==========================================================
# 3) DENSE RETRIEVER SETUP
# ==========================================================
try:
    import faiss
    import sentence_transformers
except:
    !pip install -q sentence-transformers faiss-cpu

import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

class DenseIndex:
    def __init__(self, model_id):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading Dense Model {model_id} on {self.device}...")
        self.model = SentenceTransformer(model_id, device=self.device)
        self.index = None
        self.ids = []

    def build(self, texts, ids):
        self.ids = ids
        emb = self.model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def search(self, query, top_k):
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores, idxs = self.index.search(q_emb, top_k)
        return [(self.ids[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]

# Clause-to-Control Adapter
class ClauseDenseControlAdapter:
    def __init__(self, dense_idx, clause_map, text_map):
        self.idx = dense_idx
        self.cmap = clause_map
        self.tmap = text_map
        self.last_best_clause = {} # query-specific cache

    def search(self, query, top_k=50):
        # Search more clauses to ensure we get enough unique controls
        hits = self.idx.search(query, top_k=top_k*5)
        best_score = {}
        best_clause = {}

        for cid, score in hits:
            ctl = self.cmap.get(cid)
            if not ctl: continue
            ctl = normalize_control_id(ctl)
            if score > best_score.get(ctl, -1.0):
                best_score[ctl] = score
                best_clause[ctl] = cid

        self.last_best_clause = best_clause
        ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def best_clause_text_for_control(self, ctl_id):
        cid = normalize_control_id(ctl_id)
        clid = self.last_best_clause.get(cid)
        return self.tmap.get(clid) if clid else None

def build_dense_retriever(jsonl_path):
    recs = load_clause_records_jsonl(jsonl_path, keep_kinds={"smt", "gdn"})
    ids, texts = [], []
    cmap, tmap = {}, {}
    for r in recs:
        header = f"{r['control_id']} {r['title']} ({r['kind']})"
        full = f"{header}\n{r['text']}"
        ids.append(r["clause_id"])
        texts.append(f"passage: {full}") # e5 prefix
        cmap[r["clause_id"]] = r["control_id"]
        tmap[r["clause_id"]] = full

    idx = DenseIndex(DENSE_MODEL_ID)
    idx.build(texts, ids)
    return ClauseDenseControlAdapter(idx, cmap, tmap)

def find_rank(gold, ranked_list, top_n):
    g = normalize_control_id(gold)
    for i, c in enumerate(ranked_list[:top_n], 1):
        if normalize_control_id(c) == g:
            return i
    return 0

def strict_clean(s: str) -> str:
    """Strict alnum-only key for stable exact matching."""
    return "".join(c.lower() for c in str(s) if c.isalnum())

def jaccard_overlap(a: str, b: str) -> float:
    """Token Jaccard overlap between two queries (0..1)."""
    A = set(tokenize(normalize_text(a)))
    B = set(tokenize(normalize_text(b)))
    if not A or not B: return 0.0
    return len(A & B) / float(len(A | B))

def normalize_scores_minmax(x: np.ndarray) -> np.ndarray:
    """Min-max normalize to [0,1]. Returns zeros if constant."""
    if x.size == 0: return x
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < 1e-12: return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)

def get_qur_variants_filtered(original_query: str, qur_df: pd.DataFrame,
                              max_rewrites: int = 3, jaccard_min: float = 0.15) -> list:
    """Return [original] + filtered rewrites (deduped)."""
    variants = [original_query]
    if qur_df is None or qur_df.empty: return variants

    if 'strict_key' not in qur_df.columns and 'original_query' in qur_df.columns:
        qur_df['strict_key'] = qur_df['original_query'].apply(strict_clean)

    q_key = strict_clean(original_query)
    matches = qur_df[qur_df['strict_key'] == q_key] if 'strict_key' in qur_df.columns else pd.DataFrame()

    if matches.empty: return variants

    rewrites = []
    for r in matches['rewritten_query'].tolist():
        r = str(r).strip()
        if len(r) < 6: continue
        if jaccard_overlap(original_query, r) < jaccard_min: continue
        rewrites.append(r)

    variants.extend(rewrites[:max_rewrites])
    return list(dict.fromkeys(variants)) # stable dedupe

def rrf_fuse(lists_of_cids, k=60, weights=None):
    """Reciprocal Rank Fusion with optional list-weights."""
    scores = defaultdict(float)
    if weights is None: weights = [1.0] * len(lists_of_cids)
    for w, lst in zip(weights, lists_of_cids):
        for r, cid in enumerate(lst, 1):
            scores[cid] += float(w) * (1.0 / (k + r))
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# --- Core Logic Helpers (The "Brains") ---

def s5_hybrid_rrf_details(query: str, bm25_retriever, dense_retriever,
                          bm25_control_ids: list, candidate_set_size: int = 50, rrf_k: int = 60) -> dict:
    """Unified S5 logic: Computes RRF + Agreement + Margin Stats."""
    # 1. BM25
    q_tokens = tokenize(normalize_text(query))
    bm25_scores = bm25_retriever.get_scores(q_tokens)
    bm25_idxs = bm25_retriever.get_top_n(q_tokens, n=candidate_set_size)
    bm25_cids = [normalize_control_id(bm25_control_ids[i]).upper() for i in bm25_idxs]

    # 2. Dense
    try: dense_hits = dense_retriever.search(query, top_k=candidate_set_size)
    except: dense_hits = []
    dense_cids = [normalize_control_id(str(c)).upper() for c, _ in dense_hits]

    # 3. RRF
    rrf_scores = defaultdict(float)
    for r, cid in enumerate(bm25_cids, 1): rrf_scores[cid] += 1.0 / (rrf_k + r)
    for r, cid in enumerate(dense_cids, 1): rrf_scores[cid] += 1.0 / (rrf_k + r)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    fused_cids = [c for c, _ in fused]

    # 4. Stats (ensure floats)
    bm25_top1 = bm25_cids[0] if bm25_cids else ""
    dense_top1 = dense_cids[0] if dense_cids else ""

    bm25_s1 = float(bm25_scores[bm25_idxs[0]]) if bm25_idxs else 0.0
    bm25_s2 = float(bm25_scores[bm25_idxs[1]]) if len(bm25_idxs) > 1 else 0.0
    dense_s1 = float(dense_hits[0][1]) if dense_hits else 0.0
    dense_s2 = float(dense_hits[1][1]) if len(dense_hits) > 1 else 0.0

    fused_s1 = float(fused[0][1]) if len(fused) > 0 else 0.0
    fused_s2 = float(fused[1][1]) if len(fused) > 1 else 0.0
    margin_ratio = (fused_s1 - fused_s2) / max(fused_s1, 1e-9)
    agreement = int(bool(bm25_top1) and (bm25_top1 == dense_top1))

    return {
        "bm25_top1": bm25_top1, "bm25_s1": bm25_s1, "bm25_s2": bm25_s2,
        "dense_top1": dense_top1, "dense_s1": dense_s1, "dense_s2": dense_s2,
        "fused_cids": fused_cids, "rrf_scores": rrf_scores,
        "agreement": agreement, "margin_ratio": margin_ratio
    }

def s7_heavy_retrieve_details(
    original_query,
    bm25_retriever,
    dense_retriever,
    bm25_control_ids,
    cross_enc,
    id_to_text_map,
    qur_df,
    candidate_set_size=50,
    rrf_k=60,
    max_rewrites=3,
    rewrite_weight=0.25,
    rewrite_jaccard_min=0.15,
    rerank_alpha=0.65,
    rerank_apply_min_margin_ratio=0.15,   # no-harm gate threshold (post-rerank)
    rerank_skip_enabled=True,             # performance + accuracy gate (pre-rerank)
    rerank_skip_min_base_margin_ratio=0.25,
    rerank_skip_require_top1_agreement=True,
):
    """
    S7 logic: Weighted RRF + Safe Rerank Blending + (1) Pre-rerank Skip Gate + (2) No-Harm Post-rerank Gate.

    (1) Pre-rerank Skip Gate (performance + accuracy):
        - Skip cross-encoder reranking when base fused ranking looks confident.
        - Uses cheap signals (base margin ratio; optional BM25/Dense top1 agreement).

    (2) No-Harm Post-rerank Gate (accuracy):
        - If reranker flips top1 but has low margin, keep base ordering.
    """

    variants = get_qur_variants_filtered(original_query, qur_df, max_rewrites, rewrite_jaccard_min)
    weights = [1.0] + [rewrite_weight] * (len(variants) - 1)

    first_variant = variants[0] if variants else original_query
    bm25_top1 = ""
    dense_top1 = ""

    # 1) Weighted RRF retrieval
    rrf_scores = defaultdict(float)
    best_text_cache = {}

    for q, w in zip(variants, weights):
        # BM25
        q_toks = tokenize(normalize_text(q))
        idxs = bm25_retriever.get_top_n(q_toks, n=candidate_set_size)

        if (q == first_variant) and idxs:
            try:
                bm25_top1 = normalize_control_id(bm25_control_ids[idxs[0]]).upper()
            except Exception:
                bm25_top1 = ""

        for r, i in enumerate(idxs, 1):
            cid = normalize_control_id(bm25_control_ids[i]).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if cid not in best_text_cache:
                best_text_cache[cid] = id_to_text_map.get(cid, "")

        # Dense
        try:
            hits = dense_retriever.search(q, top_k=candidate_set_size)
        except Exception:
            hits = []

        if (q == first_variant) and hits:
            try:
                dense_top1 = normalize_control_id(str(hits[0][0])).upper()
            except Exception:
                dense_top1 = ""

        for r, (raw_cid, _) in enumerate(hits, 1):
            cid = normalize_control_id(str(raw_cid)).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if hasattr(dense_retriever, "best_clause_text_for_control"):
                t = dense_retriever.best_clause_text_for_control(cid)
                if t:
                    best_text_cache[cid] = t

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:candidate_set_size]
    cids_base = [c for c, _ in fused]

    if not cids_base:
        return {
            "final_ranked_cids": [],
            "meta": {
                "note": "no_candidates",
                "reranker_called": False,
                "rerank_applied": False,
            }
        }

    # 2) Build cross-encoder pairs (only for controls with available text)
    pairs = []
    valid_cids = []
    for cid in cids_base:
        txt = best_text_cache.get(cid) or id_to_text_map.get(cid, "")
        if txt:
            pairs.append([original_query, txt])
            valid_cids.append(cid)

    base_ranked_cids = valid_cids[:] if valid_cids else cids_base[:]

    # Base scores and margin (cheap confidence signal)
    base_scores = np.array([rrf_scores.get(c, 0.0) for c in base_ranked_cids], dtype=np.float32)
    base_n = normalize_scores_minmax(base_scores)

    if len(base_n) >= 2:
        bs1 = float(base_n[0])
        bs2 = float(base_n[1])
        base_margin_ratio = (bs1 - bs2) / max(bs1, 1e-9)
    else:
        base_margin_ratio = 1.0

    base_top1 = base_ranked_cids[0] if base_ranked_cids else ""

    top1_agree = int(bool(bm25_top1) and bool(dense_top1) and (bm25_top1 == dense_top1) and (bm25_top1 == base_top1))

    # 3) Pre-rerank skip gate (performance + accuracy)
    if bool(rerank_skip_enabled):
        require_agree_ok = (not bool(rerank_skip_require_top1_agreement)) or (top1_agree == 1)
        if require_agree_ok and (base_margin_ratio >= float(rerank_skip_min_base_margin_ratio)):
            return {
                "final_ranked_cids": base_ranked_cids,
                "meta": {
                    "num_variants": len(variants),
                    "reranker_called": False,
                    "rerank_applied": False,
                    "skip_reason": "base_confident",
                    "rerank_alpha": float(rerank_alpha),
                    "rerank_apply_min_margin_ratio": float(rerank_apply_min_margin_ratio),
                    "rerank_skip_min_base_margin_ratio": float(rerank_skip_min_base_margin_ratio),
                    "rerank_skip_require_top1_agreement": bool(rerank_skip_require_top1_agreement),
                    "bm25_top1": bm25_top1,
                    "dense_top1": dense_top1,
                    "top1_agree": int(top1_agree),
                    "base_top1": base_top1,
                    "rerank_top1": base_top1,
                    "final_top1": base_top1,
                    "base_margin_ratio": float(base_margin_ratio),
                    "rerank_margin_ratio": -1.0,
                    "final_margin_ratio": float(base_margin_ratio),
                }
            }

    # If we have no text, fallback to base order (cannot rerank)
    if not pairs:
        return {
            "final_ranked_cids": cids_base,
            "meta": {
                "note": "no_text_for_rerank",
                "num_variants": len(variants),
                "reranker_called": False,
                "rerank_applied": False,
                "bm25_top1": bm25_top1,
                "dense_top1": dense_top1,
                "top1_agree": int(top1_agree),
                "base_top1": cids_base[0] if cids_base else "",
                "base_margin_ratio": float(base_margin_ratio),
            }
        }

    # 4) Cross-encoder scoring (expensive)
    pred = cross_enc.predict(pairs, show_progress_bar=False)
    pred_n = normalize_scores_minmax(np.array(pred, dtype=np.float32))

    # Safe blending score
    final_scores = float(rerank_alpha) * base_n[:len(pred_n)] + (1.0 - float(rerank_alpha)) * pred_n

    # 5) Reranked order by blended score
    rerank_order = np.argsort(-final_scores)
    reranked_cids = [valid_cids[i] for i in rerank_order]

    # Compute rerank margin ratio (top1 vs top2)
    if len(rerank_order) >= 2:
        fs1 = float(final_scores[rerank_order[0]])
        fs2 = float(final_scores[rerank_order[1]])
        rerank_margin_ratio = (fs1 - fs2) / max(fs1, 1e-9)
    else:
        rerank_margin_ratio = 1.0

    rerank_top1 = reranked_cids[0] if reranked_cids else ""

    # 6) No-Harm Post-rerank Gate
    rerank_applied = True
    final_ranked_cids = reranked_cids

    if (rerank_top1 != base_top1) and (rerank_margin_ratio < float(rerank_apply_min_margin_ratio)):
        rerank_applied = False
        final_ranked_cids = base_ranked_cids

    final_margin_ratio = rerank_margin_ratio if rerank_applied else base_margin_ratio
    final_top1 = final_ranked_cids[0] if final_ranked_cids else ""

    return {
        "final_ranked_cids": final_ranked_cids,
        "meta": {
            "num_variants": len(variants),
            "reranker_called": True,
            "rerank_applied": bool(rerank_applied),
            "rerank_alpha": float(rerank_alpha),
            "rerank_apply_min_margin_ratio": float(rerank_apply_min_margin_ratio),
            "rerank_skip_min_base_margin_ratio": float(rerank_skip_min_base_margin_ratio),
            "rerank_skip_require_top1_agreement": bool(rerank_skip_require_top1_agreement),
            "bm25_top1": bm25_top1,
            "dense_top1": dense_top1,
            "top1_agree": int(top1_agree),
            "base_top1": base_top1,
            "rerank_top1": rerank_top1,
            "final_top1": final_top1,
            "base_margin_ratio": float(base_margin_ratio),
            "rerank_margin_ratio": float(rerank_margin_ratio),
            "final_margin_ratio": float(final_margin_ratio),
        }
    }

def system_s1_bm25(q, gold, bm25, doc_ids, top_n=10):
    q_toks = tokenize(normalize_text(q))
    idxs = bm25.get_top_n(q_toks, n=top_n)
    retrieved = [normalize_control_id(doc_ids[i]) for i in idxs]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S1"}}

def system_s2_dense(q, gold, dense, top_n=10):
    hits = dense.search(q, top_k=top_n)
    retrieved = [normalize_control_id(c) for c, s in hits]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S2"}}

def system_s5_hybrid(q, gold, bm25, doc_ids, dense, top_n=10, candidate_k=50, rrf_k=60):
    """S5 baseline Hybrid RRF. S5 and S8 FAST path are mathematically identical."""
    details = s5_hybrid_rrf_details(q, bm25, dense, doc_ids, candidate_set_size=candidate_k, rrf_k=rrf_k)
    final = details["fused_cids"][:top_n]
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": {"sys": "S5"}}

def system_s6_rerank(q, gold, bm25, doc_ids, dense, id_to_text_map, top_n=10):
    res_s5 = system_s5_hybrid(q, gold, bm25, doc_ids, dense, top_n=CANDIDATE_SET_SIZE, candidate_k=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT)
    cands = res_s5["retrieved_ids"]

    pairs = []
    valid_cands = []
    for cid in cands:
        txt = dense.best_clause_text_for_control(cid) if hasattr(dense, "best_clause_text_for_control") else None
        if not txt: txt = id_to_text_map.get(normalize_control_id(cid), "")
        if txt:
            pairs.append([q, txt])
            valid_cands.append(cid)

    if not pairs: return {"rank": 0, "retrieved_ids": [], "system_meta": {"sys": "S6", "note": "no_text"}}

    scores = cross_encoder.predict(pairs)
    reranked = [c for _, c in sorted(zip(scores, valid_cands), reverse=True)]
    final = reranked[:top_n]
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": {"sys": "S6"}}

def system_s7_compliancegpt(q, gold, bm25, doc_ids, dense, id_to_text_map, qur_df, top_n=10):
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, doc_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        max_rewrites=S7_MAX_REWRITES, rewrite_weight=S7_REWRITE_WEIGHT, rewrite_jaccard_min=S7_REWRITE_JACCARD_MIN, rerank_alpha=S7_RERANK_ALPHA,
        rerank_apply_min_margin_ratio=S7_RERANK_APPLY_MIN_MARGIN_RATIO,
        rerank_skip_enabled=S7_RERANK_SKIP_ENABLED,
        rerank_skip_min_base_margin_ratio=S7_RERANK_SKIP_MIN_BASE_MARGIN_RATIO,
        rerank_skip_require_top1_agreement=S7_RERANK_SKIP_REQUIRE_TOP1_AGREEMENT,
    )
    final = heavy["final_ranked_cids"][:top_n]
    meta = heavy.get("meta", {})
    meta["sys"] = "S7"
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": meta}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.4 MB/s eta 0:00:00


In [4]:
# ==========================================================
# 1) LOAD CCS + BUILD INDICES + LOAD QUR
# ==========================================================
import torch
from sentence_transformers import CrossEncoder

def load_for_dataset(dataset: str):
    dataset = dataset.lower().strip()
    if dataset == "rev5":
        catalog_path = REV5_CATALOG_PATH
        qur_path = QUR_CSV_REV5
    elif dataset == "rev4":
        catalog_path = REV4_CATALOG_PATH
        qur_path = QUR_CSV_REV4
    else:
        raise ValueError(f"Unsupported dataset: {dataset}")

    print(f"--- Loading CCS: {catalog_path} ---")
    clause_records = load_clause_records_jsonl(catalog_path, keep_kinds=("smt","gdn"))
    print(f"Loaded clauses: {len(clause_records)}")

    print("--- Building BM25 (control-level docs) ---")
    control_docs = build_control_docs_from_clauses(clause_records)
    bm25, bm25_control_ids = build_bm25(control_docs)
    print(f"BM25 docs (controls): {len(bm25_control_ids)}")

    print("--- Building Dense (clause-level) ---")
    dense = build_dense_retriever(catalog_path)

    print("--- Loading Cross-Encoder reranker ---")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cross_enc = CrossEncoder(RERANKER_MODEL_ID, device=device)

    print("--- Building fallback control text map (for reranker) ---")
    id_to_text = build_control_fallback_text_map(clause_records)

    print("--- Loading QUR rewrites CSV ---")
    qur_df = pd.read_csv(qur_path, encoding="ISO-8859-1")
    print(f"QUR rows: {len(qur_df)} | unique queries: {qur_df['query_id'].nunique() if 'query_id' in qur_df.columns else 'n/a'}")

    return clause_records, bm25, bm25_control_ids, dense, cross_enc, id_to_text, qur_df

clause_records, bm25, bm25_control_ids, dense, cross_encoder, id_to_text_map, qur_df = load_for_dataset(DATASET)

print("System ready.")


--- Loading CCS: /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl ---
Loaded clauses: 3136
--- Building BM25 (control-level docs) ---
BM25 docs (controls): 1013
--- Building Dense (clause-level) ---
Loading Dense Model intfloat/e5-small-v2 on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

--- Building fallback control text map (for reranker) ---
--- Loading QUR rewrites CSV ---
QUR rows: 300 | unique queries: 100
System ready.


In [5]:
# ==========================================================
# 2) SINGLE-QUERY PATH CHECK (prints S1/S2/S5/S6/S7)
# ==========================================================
def pick_query(qur_df: pd.DataFrame, use_single: bool):
    if use_single:
        df = qur_df[qur_df["query_id"] == int(SINGLE_QUERY_ID)]
        if df.empty:
            raise ValueError(f"No rows for query_id={SINGLE_QUERY_ID}")
        q = str(df.iloc[0]["original_query"])
        qid = int(df.iloc[0]["query_id"])
        return qid, q
    # sample unique queries
    uq = qur_df.drop_duplicates(subset=["query_id"]).head(int(NUM_QUERIES))
    qid = int(uq.iloc[0]["query_id"])
    q = str(uq.iloc[0]["original_query"])
    return qid, q

qid, q = pick_query(qur_df, use_single=True)  # always debug on one query

print("===== SINGLE QUERY =====")
print(f"qid: {qid}")
print(f"q: {q}")

# rewrites raw
rw = qur_df[qur_df["query_id"] == qid][["rewrite_rank","rewritten_query"]].sort_values("rewrite_rank")
print("\n===== REWRITES (raw) =====")
display(rw)

variants = get_qur_variants_filtered(q, qur_df, max_rewrites=S7_MAX_REWRITES, jaccard_min=S7_REWRITE_JACCARD_MIN)
print("\n===== S7 VARIANTS (after filter) =====")
print(f"num_variants = {len(variants)}")
for i,v in enumerate(variants):
    print(f"  [{i}] {v}")

# S6 candidate pool is S5 top-50 (before rerank text filtering)
res_s5 = system_s5_hybrid(q, gold="", bm25=bm25, doc_ids=bm25_control_ids, dense=dense, top_n=CANDIDATE_SET_SIZE, candidate_k=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT)
pool = res_s5["retrieved_ids"]
print("\n===== S6 PRE-RERANK CANDIDATE POOL =====")
print(f"pool_size (before text filtering) = {len(pool)}")
print(f"top10_pool = {pool[:10]}")

def run_and_print(label, fn):
    out = fn()
    print(f"\n===== {label} =====")
    print("top10 =", out["retrieved_ids"][:10])
    print("meta  =", json.dumps(out["system_meta"], indent=2))
    return out

run_and_print("S1", lambda: system_s1_bm25(q, gold="", bm25=bm25, doc_ids=bm25_control_ids, top_n=10))
run_and_print("S2", lambda: system_s2_dense(q, gold="", dense=dense, top_n=10))
run_and_print("S5", lambda: system_s5_hybrid(q, gold="", bm25=bm25, doc_ids=bm25_control_ids, dense=dense, top_n=10, candidate_k=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT))
run_and_print("S6", lambda: system_s6_rerank(q, gold="", bm25=bm25, doc_ids=bm25_control_ids, dense=dense, id_to_text_map=id_to_text_map, top_n=10))

# S7: typical (gated) vs worst (always rerank)
def run_s7(q, gate_min_margin_ratio):
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, bm25_control_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        max_rewrites=S7_MAX_REWRITES, rewrite_weight=S7_REWRITE_WEIGHT, rewrite_jaccard_min=S7_REWRITE_JACCARD_MIN,
        rerank_alpha=S7_RERANK_ALPHA, rerank_apply_min_margin_ratio=gate_min_margin_ratio,
        rerank_skip_enabled=S7_RERANK_SKIP_ENABLED,
        rerank_skip_min_base_margin_ratio=S7_RERANK_SKIP_MIN_BASE_MARGIN_RATIO,
        rerank_skip_require_top1_agreement=S7_RERANK_SKIP_REQUIRE_TOP1_AGREEMENT,
    )
    final = heavy["final_ranked_cids"][:10]
    meta = heavy.get("meta", {})
    meta["sys"] = "S7"
    return {"retrieved_ids": final, "system_meta": meta}

run_and_print("S7 (gated)", lambda: run_s7(q, S7_RERANK_APPLY_MIN_MARGIN_RATIO))
run_and_print("S7 (worst / always rerank)", lambda: run_s7(q, S7_RERANK_APPLY_MIN_MARGIN_RATIO_WORST))


===== SINGLE QUERY =====
qid: 1
q: When must account managers and the designated personnel be notified about user account changes?

===== REWRITES (raw) =====


,rewrite_rank,rewritten_query
0,0,When do account managers and designated person...
1,1,What are the requirements for notifying accoun...
2,2,Under what circumstances should account manage...



===== S7 VARIANTS (after filter) =====
num_variants = 4
  [0] When must account managers and the designated personnel be notified about user account changes?
  [1] When do account managers and designated personnel need to be informed about changes to user accounts?
  [2] What are the requirements for notifying account managers and designated personnel about user account modifications?
  [3] Under what circumstances should account managers and designated personnel be made aware of changes in user accounts?

===== S6 PRE-RERANK CANDIDATE POOL =====
pool_size (before text filtering) = 50
top10_pool = ['AC-2', 'AC-9.3', 'AC-2.1', 'AC-2.7', 'AC-2.11', 'CM-6.2', 'CM-3.1', 'AC-2.12', 'IA-4.1', 'AC-2.4']

===== S1 =====
top10 = ['AC-2', 'AC-9.3', 'AC-2.1', 'IA-4.1', 'IA-12.1', 'AC-2.11', 'AC-2.7', 'AC-6.2', 'IA-5.1', 'IA-12.2']
meta  = {
  "sys": "S1"
}

===== S2 =====
top10 = ['AC-9.3', 'AC-2', 'AU-12.3', 'AC-2.1', 'AC-2.7', 'AC-6.7', 'AC-2.11', 'PS-5', 'CM-3.1', 'SA-10.7']
meta  = {
  "sys"

{'retrieved_ids': ['AC-2.1',
  'AC-9.3',
  'AC-2',
  'AC-2.7',
  'AC-2.11',
  'CM-3.1',
  'CM-6.2',
  'IA-4.1',
  'AC-2.12',
  'AC-2.2'],
 'system_meta': {'num_variants': 4,
  'reranker_called': True,
  'rerank_applied': True,
  'rerank_alpha': 0.65,
  'rerank_apply_min_margin_ratio': 0.0,
  'rerank_skip_min_base_margin_ratio': 0.1,
  'rerank_skip_require_top1_agreement': False,
  'bm25_top1': 'AC-2',
  'dense_top1': 'AC-9.3',
  'top1_agree': 0,
  'base_top1': 'AC-2',
  'rerank_top1': 'AC-2.1',
  'final_top1': 'AC-2.1',
  'base_margin_ratio': 0.02119690179824829,
  'rerank_margin_ratio': 0.01012248167823262,
  'final_margin_ratio': 0.01012248167823262,
  'sys': 'S7'}}

In [6]:
# ==========================================================
# 3) PERFORMANCE BENCHMARK (latency)
# ==========================================================
import statistics as stats
import hashlib

def _stable_hash_int(s: str) -> int:
    return int(hashlib.sha256(s.encode("utf-8")).hexdigest()[:16], 16)

def get_query_list(qur_df: pd.DataFrame) -> List[Dict[str, Any]]:
    if USE_SINGLE_QUERY:
        df1 = qur_df[qur_df["query_id"] == int(SINGLE_QUERY_ID)]
        if df1.empty:
            raise ValueError(f"No rows for query_id={SINGLE_QUERY_ID}")
        row = df1.iloc[0]
        return [{"query_id": int(row["query_id"]), "query": str(row["original_query"])}]

    uq = qur_df.drop_duplicates(subset=["query_id"])[["query_id", "original_query"]].copy()
    uq["query_id"] = uq["query_id"].astype(int)
    uq["original_query"] = uq["original_query"].astype(str)

    n = min(int(NUM_QUERIES), int(len(uq)))
    if QUERY_SAMPLE_METHOD == "random":
        uq = uq.sort_values("query_id")
        sampled = uq.sample(n=n, random_state=int(SAMPLE_SEED), replace=False)
        sampled = sampled.sort_values("query_id")
    elif QUERY_SAMPLE_METHOD == "hash":
        uq = uq.sort_values("query_id")
        uq["h"] = uq["original_query"].map(lambda s: _stable_hash_int(str(s)))
        sampled = uq.sort_values("h").head(n).sort_values("query_id")
    else:
        raise ValueError(f"Unknown QUERY_SAMPLE_METHOD={QUERY_SAMPLE_METHOD}")

    out: List[Dict[str, Any]] = []
    for _, row in sampled.iterrows():
        out.append({"query_id": int(row["query_id"]), "query": str(row["original_query"])})
    return out

queries = get_query_list(qur_df)
print(f"Benchmark queries: {len(queries)}")

def gpu_sync() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def time_once(fn) -> Tuple[float, Dict[str, Any]]:
    gpu_sync()
    t0 = time.perf_counter()
    out = fn()
    gpu_sync()
    t1 = time.perf_counter()
    return (t1 - t0) * 1000.0, out

# System runners (return top10 + meta)
def run_s1(q: str) -> Dict[str, Any]:
    return system_s1_bm25(q, gold="", bm25=bm25, doc_ids=bm25_control_ids, top_n=10)

def run_s2(q: str) -> Dict[str, Any]:
    return system_s2_dense(q, gold="", dense=dense, top_n=10)

def run_s5(q: str) -> Dict[str, Any]:
    return system_s5_hybrid(
        q, gold="", bm25=bm25, doc_ids=bm25_control_ids, dense=dense,
        top_n=10, candidate_k=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT
    )

def run_s6(q: str) -> Dict[str, Any]:
    return system_s6_rerank(
        q, gold="", bm25=bm25, doc_ids=bm25_control_ids, dense=dense,
        id_to_text_map=id_to_text_map, top_n=10
    )

def run_s7_gated(q: str) -> Dict[str, Any]:
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, bm25_control_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        max_rewrites=S7_MAX_REWRITES, rewrite_weight=S7_REWRITE_WEIGHT, rewrite_jaccard_min=S7_REWRITE_JACCARD_MIN,
        rerank_alpha=S7_RERANK_ALPHA,
        rerank_apply_min_margin_ratio=S7_RERANK_APPLY_MIN_MARGIN_RATIO,
        rerank_skip_enabled=S7_RERANK_SKIP_ENABLED,
        rerank_skip_min_base_margin_ratio=S7_RERANK_SKIP_MIN_BASE_MARGIN_RATIO,
        rerank_skip_require_top1_agreement=S7_RERANK_SKIP_REQUIRE_TOP1_AGREEMENT,
    )
    final = heavy["final_ranked_cids"][:10]
    meta = heavy.get("meta", {}) or {}
    meta["sys"] = "S7_gated"
    return {"rank": 0, "retrieved_ids": final, "system_meta": meta}

def run_s7_worst(q: str) -> Dict[str, Any]:
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, bm25_control_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        max_rewrites=S7_MAX_REWRITES, rewrite_weight=S7_REWRITE_WEIGHT, rewrite_jaccard_min=S7_REWRITE_JACCARD_MIN,
        rerank_alpha=S7_RERANK_ALPHA,
        rerank_apply_min_margin_ratio=S7_RERANK_APPLY_MIN_MARGIN_RATIO_WORST,
        rerank_skip_enabled=False,
    )
    final = heavy["final_ranked_cids"][:10]
    meta = heavy.get("meta", {}) or {}
    meta["sys"] = "S7_worst"
    return {"rank": 0, "retrieved_ids": final, "system_meta": meta}

SYSTEMS = [
    ("S1_bm25", run_s1),
    ("S2_dense", run_s2),
    ("S5_hybrid_rrf", run_s5),
    ("S6_hybrid_rerank", run_s6),
    ("S7_gated", run_s7_gated),
    ("S7_worst_always_rerank", run_s7_worst),
]

# Warmup (important for GPU + caching)
print("Warmup...")
warm_q = queries[0]["query"]
for _, fn in SYSTEMS:
    _ = fn(warm_q)

rows: List[Dict[str, Any]] = []
run_records: List[Dict[str, Any]] = []
s7_called_flags: List[bool] = []
s7_applied_flags: List[bool] = []
s7_skip_reason_counts: Dict[str, int] = defaultdict(int)

for name, fn in SYSTEMS:
    times: List[float] = []
    for qrec in queries:
        q = str(qrec["query"])
        qid = int(qrec["query_id"])
        for rep_i in range(int(REPEATS_PER_QUERY)):
            ms, out = time_once(lambda: fn(q))
            times.append(ms)

            if bool(SAVE_RUN_LOG):
                rec: Dict[str, Any] = {
                    "system": str(name),
                    "query_id": int(qid),
                    "repeat_i": int(rep_i),
                    "latency_ms": float(ms),
                }
                if bool(RUN_LOG_INCLUDE_S7_META) and isinstance(out, dict):
                    meta = out.get("system_meta", {}) or {}
                    if str(name).startswith("S7"):
                        rec.update({
                            "reranker_called": bool(meta.get("reranker_called", False)),
                            "rerank_applied": bool(meta.get("rerank_applied", False)),
                            "skip_reason": str(meta.get("skip_reason", "")),
                            "top1_agree": int(meta.get("top1_agree", 0)) if meta.get("top1_agree", None) is not None else 0,
                            "base_margin_ratio": float(meta.get("base_margin_ratio", -1.0)) if meta.get("base_margin_ratio", None) is not None else -1.0,
                            "rerank_margin_ratio": float(meta.get("rerank_margin_ratio", -1.0)) if meta.get("rerank_margin_ratio", None) is not None else -1.0,
                            "final_margin_ratio": float(meta.get("final_margin_ratio", -1.0)) if meta.get("final_margin_ratio", None) is not None else -1.0,
                            "base_top1": str(meta.get("base_top1", "")),
                            "rerank_top1": str(meta.get("rerank_top1", "")),
                            "final_top1": str(meta.get("final_top1", "")),
                        })
                run_records.append(rec)

            if name == "S7_gated" and isinstance(out, dict):
                meta = out.get("system_meta", {}) or {}
                flag_called = bool(meta.get("reranker_called", False))
                flag_applied = bool(meta.get("rerank_applied", False))
                s7_called_flags.append(flag_called)
                s7_applied_flags.append(flag_applied)
                if not flag_called:
                    reason = str(meta.get("skip_reason", "")) or "skipped_no_reason"
                    s7_skip_reason_counts[reason] = int(s7_skip_reason_counts.get(reason, 0)) + 1

    times_np = np.array(times, dtype=np.float32)
    rows.append({
        "system": str(name),
        "n_runs": int(times_np.size),
        "mean_ms": float(times_np.mean()),
        "p50_ms": float(np.percentile(times_np, 50)),
        "p95_ms": float(np.percentile(times_np, 95)),
        "min_ms": float(times_np.min()),
        "max_ms": float(times_np.max()),
    })

df = pd.DataFrame(rows).sort_values("mean_ms")
display(df)

run_log_df = pd.DataFrame(run_records) if bool(SAVE_RUN_LOG) else pd.DataFrame([])
if bool(SAVE_RUN_LOG):
    display(run_log_df.head(10))

if s7_called_flags:
    called_rate = float(sum(s7_called_flags) / len(s7_called_flags))
    applied_rate = float(sum(s7_applied_flags) / len(s7_applied_flags)) if s7_applied_flags else 0.0
    print(f"\nS7 gated: reranker_called_rate = {called_rate:.3f} ({sum(s7_called_flags)}/{len(s7_called_flags)})")
    print(f"S7 gated: rerank_applied_rate = {applied_rate:.3f} ({sum(s7_applied_flags)}/{len(s7_applied_flags)})")
    if s7_skip_reason_counts:
        print("S7 gated: skip_reason_counts =", dict(s7_skip_reason_counts))


Benchmark queries: 20
Warmup...


,system,n_runs,mean_ms,p50_ms,p95_ms,min_ms,max_ms
0,S1_bm25,100,8.6586,8.3799,12.1480,4.2630,14.5470
1,S2_dense,100,13.0646,12.3125,17.3595,11.6571,19.0787
2,S5_hybrid_rrf,100,30.0895,29.7993,36.8857,21.8507,38.0392
4,S7_gated,100,411.1000,585.4338,716.1569,78.4888,762.2572
3,S6_hybrid_rerank,100,600.5952,626.2180,646.6352,497.7229,659.5706
5,S7_worst_always_rerank,100,679.1351,690.3068,730.8356,545.5856,746.8328


,system,query_id,repeat_i,latency_ms,reranker_called,rerank_applied,skip_reason,top1_agree,base_margin_ratio,rerank_margin_ratio,final_margin_ratio,base_top1,rerank_top1,final_top1
0,S1_bm25,2,0,8.1091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S1_bm25,2,1,7.4050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S1_bm25,2,2,7.2791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S1_bm25,2,3,7.7737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S1_bm25,2,4,8.0205,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,S1_bm25,4,0,6.2889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,S1_bm25,4,1,5.9607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,S1_bm25,4,2,5.7163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,S1_bm25,4,3,5.5071,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,S1_bm25,4,4,5.4087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



S7 gated: reranker_called_rate = 0.550 (55/100)
S7 gated: rerank_applied_rate = 0.500 (50/100)
S7 gated: skip_reason_counts = {'base_confident': 45}


In [7]:
# ==========================================================
# 4) SAVE ARTIFACTS (CSV + Markdown + metadata)
# ==========================================================
from datetime import datetime
import json
import os

OUTPUT_DIR = "/content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/performance_benchmark/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tag = f"{DATASET}_{len(queries)}q"
csv_out = os.path.join(OUTPUT_DIR, f"retriever_performance_benchmark_{tag}.csv")
runs_out = os.path.join(OUTPUT_DIR, f"retriever_performance_benchmark_runs_{tag}.csv")
md_out = os.path.join(OUTPUT_DIR, f"retriever_performance_benchmark_table_{tag}.md")
meta_out = os.path.join(OUTPUT_DIR, f"retriever_performance_benchmark_meta_{tag}.json")

df.to_csv(csv_out, index=False)

if "run_log_df" in globals() and hasattr(run_log_df, "to_csv") and (len(run_log_df) > 0):
    run_log_df.to_csv(runs_out, index=False)
else:
    runs_out = ""

try:
    md_table = df.to_markdown(index=False)
except Exception:
    md_table = df.to_string(index=False)

with open(md_out, "w", encoding="utf-8") as f:
    f.write(md_table + "\n")

meta: Dict[str, Any] = {
    "timestamp_iso": datetime.utcnow().isoformat() + "Z",
    "dataset": DATASET,
    "num_queries": int(len(queries)),
    "repeats_per_query": int(REPEATS_PER_QUERY),
    "top_k": 10,
    "candidate_set_size": int(CANDIDATE_SET_SIZE),
    "rrf_k_default": int(RRF_K_DEFAULT),
    "dense_model_id": str(DENSE_MODEL_ID),
    "reranker_model_id": str(RERANKER_MODEL_ID),
    "s7_max_rewrites": int(S7_MAX_REWRITES),
    "s7_rewrite_weight": float(S7_REWRITE_WEIGHT),
    "s7_rewrite_jaccard_min": float(S7_REWRITE_JACCARD_MIN),
    "s7_rerank_alpha": float(S7_RERANK_ALPHA),
    "s7_rerank_apply_min_margin_ratio": float(S7_RERANK_APPLY_MIN_MARGIN_RATIO),
    "s7_rerank_skip_enabled": bool(S7_RERANK_SKIP_ENABLED),
    "s7_rerank_skip_min_base_margin_ratio": float(S7_RERANK_SKIP_MIN_BASE_MARGIN_RATIO),
    "s7_rerank_skip_require_top1_agreement": bool(S7_RERANK_SKIP_REQUIRE_TOP1_AGREEMENT),
    "s7_rerank_apply_min_margin_ratio_worst": float(S7_RERANK_APPLY_MIN_MARGIN_RATIO_WORST),
    "summary_csv": str(csv_out),
    "runs_csv": str(runs_out) if runs_out else "",
    "table_md": str(md_out),
}

if "s7_called_flags" in globals() and s7_called_flags:
    meta["s7_reranker_called_rate"] = float(sum(s7_called_flags) / len(s7_called_flags))
    meta["s7_reranker_called_count"] = int(sum(s7_called_flags))
    meta["s7_total_runs"] = int(len(s7_called_flags))

if "s7_applied_flags" in globals() and s7_applied_flags:
    meta["s7_rerank_applied_rate"] = float(sum(s7_applied_flags) / len(s7_applied_flags))
    meta["s7_rerank_applied_count"] = int(sum(s7_applied_flags))

if "s7_skip_reason_counts" in globals() and s7_skip_reason_counts:
    meta["s7_skip_reason_counts"] = {str(k): int(v) for k, v in s7_skip_reason_counts.items()}

with open(meta_out, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:")
print("  ", csv_out)
if runs_out:
    print("  ", runs_out)
print("  ", md_out)
print("  ", meta_out)


/tmp/ipython-input-1138339444.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_iso": datetime.utcnow().isoformat() + "Z",


Saved:
   /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/performance_benchmark/output/retriever_performance_benchmark_rev5_20q.csv
   /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/performance_benchmark/output/retriever_performance_benchmark_runs_rev5_20q.csv
   /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/performance_benchmark/output/retriever_performance_benchmark_table_rev5_20q.md
   /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/performance_benchmark/output/retriever_performance_benchmark_meta_rev5_20q.json
